# 01 — Entity Matching: manual, regex, and the CDF API

Companion to [Chapter 07](../07-entity-matching.md). Run every cell yourself, in
order, and read the output before moving on. Cell order mirrors the eventual
`MatchDocuments` Function handler: **auth -> list/retrieve -> mutate/job submit ->
poll with timeout -> inspect result -> cleanup.**

Before running: set `YOURNAME` below to your own uppercase participant name.

In [ ]:
YOURNAME = "YOURNAME"  # [CHANGE] your uppercase name, e.g. "ALICE"

import os
from pathlib import Path

from cognite.client import CogniteClient
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthInteractive
from cognite.client.data_classes.data_modeling import (
    DirectRelationReference,
    NodeApply,
    NodeOrEdgeData,
    ViewId,
)

# Load repo-root .env (copy from .env.example). Bare CogniteClient() has no
# default config in Jupyter — build ClientConfig from env + interactive token.
HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in [HERE, *HERE.parents] if (p / "pyproject.toml").exists() and (p / "training").exists()),
    HERE,
)
env_path = ROOT / ".env"
assert env_path.exists(), f"Missing {env_path} — copy from .env.example"
for line in env_path.read_text(encoding="utf-8").splitlines():
    s = line.strip()
    if not s or s.startswith("#") or "=" not in s:
        continue
    k, v = s.split("=", 1)
    if " #" in v and not v.startswith(('"', "'")):
        v = v.split(" #", 1)[0].rstrip()
    os.environ[k] = v

client = CogniteClient(
    ClientConfig(
        client_name=YOURNAME+"_entity-matching-notebook",
        project=os.environ["CDF_PROJECT"],
        base_url=os.environ["CDF_URL"],
        credentials=OAuthInteractive(
            authority_url=os.environ["IDP_AUTHORITY_URL"],
            client_id=os.environ["IDP_CLIENT_ID"],
            scopes=[os.environ["IDP_SCOPES"]],
        ),
    )
)
space = f"isp_{YOURNAME}_TRN"
print(client.iam.token.inspect())


## Step 1 — list your files and assets

This is the "list/retrieve" step every technique below starts from.

In [ ]:
v_file = ViewId("cdf_cdm", "CogniteFile", "v1")
v_asset = ViewId("cdf_cdm", "CogniteAsset", "v1")

file_xids = [
    f"file_{YOURNAME}_TRN_PID_21_SEP",
    f"file_{YOURNAME}_TRN_DS_21_PA_2001A",
]
files = client.data_modeling.instances.retrieve_nodes(
    nodes=[(space, xid) for xid in file_xids], sources=[v_file],
)
assets = client.data_modeling.instances.list(
    instance_type="node", sources=[v_asset], space=space, limit=-1,
)

for f in files:
    print("file:", f.external_id, "-", f.properties.get(v_file, {}).get("name"))
print(f"\n{len(assets)} assets, e.g.:")
for a in list(assets)[:3]:
    print(" asset:", a.external_id, "-", a.properties.get(v_asset, {}).get("name"))

## Technique 1 — Manual matching

You already know the answer -- you wrote it into the file YAML in Chapter 04. This is
your ground truth for checking the other two techniques against.

In [ ]:
manual_map = {
    f"file_{YOURNAME}_TRN_PID_21_SEP": "TRN-21-SEP",
    f"file_{YOURNAME}_TRN_DS_21_PA_2001A": "21-PA-2001A",
}
print("Manual (ground truth):")
for f_xid, a_xid in manual_map.items():
    print(f"  {f_xid} -> {a_xid}")

## Technique 2 — Regex / rule-based matching

Extract a tag-shaped substring from each file's `name` property and check it
resolves to a real asset externalId in this space.

**Try renaming a file in your head to `TRN_21_SEP_PID_rev2_FINAL(1).pdf` -- does the
regex still match?** That's the format-drift failure mode called out in the chapter.

In [ ]:
import re

asset_xids = {a.external_id for a in assets}
TAG_RE = re.compile(r"(\d{2}-[A-Z]{2}-\d{4}[A-Z]?)")
AREA_RE = re.compile(r"(TRN-\d{2}-[A-Z]+)")

print("Regex:")
for f in files:
    name = f.properties.get(v_file, {}).get("name") or f.external_id
    m = TAG_RE.search(name) or AREA_RE.search(name)
    candidate = m.group(1) if m else None
    resolved = candidate if candidate in asset_xids else None
    print(f"  {name!r} -> candidate={candidate!r} resolved={resolved!r}")

## Technique 3 — CDF Entity Matching API: `fit`

For text with no clean extractable tag, train an unsupervised similarity model on
`name` vs `name`. This is the same call the `MatchDocuments` Function makes -- you
are looking directly at the job the Function will later run unattended.

**This model is created in a project-wide, global namespace -- not scoped to your
space.** You WILL delete it at the end of this notebook.

In [ ]:
model_xid = f"emp_{YOURNAME}_Datasheet_TRN"

sources = [{"id": f.external_id, "name": f.properties.get(v_file, {}).get("name") or f.external_id} for f in files]
targets = [{"id": a.external_id, "name": a.properties.get(v_asset, {}).get("name") or a.external_id} for a in assets]

# Drop a leftover model from a previous run of this notebook, if any.
try:
    client.entity_matching.delete(external_id=model_xid)
    print("deleted a stale model from a previous run")
except Exception:
    pass

model = client.entity_matching.fit(
    sources=sources,
    targets=targets,
    match_fields=[("name", "name")],
    feature_type="bigram",
    external_id=model_xid,
    name=model_xid,
)
print("submitted fit job, model id:", model.id, "status:", getattr(model, "status", None))

## Poll with a timeout -- never block forever on `.result`

`[COMMON MISTAKE]` a naive `while status != "Completed"` with no deadline can hang a
notebook (or a Function, far worse) indefinitely if a job ever gets stuck. Always poll
with a bounded deadline, exactly like this.

Fit is polled with `entity_matching.retrieve(id=...)`. Predict is different: the SDK
returns a job object — call `predict.update_status()` each loop (there is no
`retrieve_predict_job`). Give predict its **own** 300s deadline; do not reuse fit's.


In [ ]:
import time

# Fit gets its own 300s budget — do not share this deadline with predict.
deadline = time.time() + 300
while getattr(model, "status", "Completed") not in ("Completed", "Failed") and time.time() < deadline:
    time.sleep(5)
    model = client.entity_matching.retrieve(id=model.id)
    print("  status:", getattr(model, "status", None))

print("final status:", getattr(model, "status", None))
assert getattr(model, "status", None) == "Completed", (
    f"fit did not complete — status={getattr(model, 'status', None)!r}; inspect before continuing"
)


## `predict` -- score every file against every asset

In [ ]:
predict = client.entity_matching.predict(
    id=model.id, num_matches=1, sources=sources, targets=targets,
)
print("submitted predict job, job_id:", getattr(predict, "job_id", None), "status:", predict.status)

# Fresh deadline for predict. Status is refreshed via update_status() on the job
# object — there is no entity_matching.retrieve_predict_job in the SDK.
deadline = time.time() + 300
while predict.status not in ("Completed", "Failed") and time.time() < deadline:
    time.sleep(5)
    print("  status:", predict.update_status())

print("final status:", predict.status)
assert predict.status == "Completed", (
    f"predict did not complete — status={predict.status!r}; inspect before continuing"
)

result = predict.get_result()
result_items = result.get("items") if isinstance(result, dict) else []
if not result_items and isinstance(result, list):
    result_items = result

matches, below = [], []
for item in result_items:
    src = item.get("source") or item.get("sourceId") or {}
    src_id = src.get("id") if isinstance(src, dict) else src
    match_list = item.get("matches") or []
    if not match_list:
        continue
    best = match_list[0]
    score = float(best.get("score") or 0.0)
    tgt = best.get("target") or best.get("targetId") or {}
    tgt_id = tgt.get("id") if isinstance(tgt, dict) else tgt
    row = {"source": src_id, "target": tgt_id, "score": round(score, 3)}
    (matches if score >= 0.5 else below).append(row)

print("matches (score >= 0.5):")
for m in matches:
    print(" ", m)
print("below threshold:")
for b in below:
    print(" ", b)


## Apply the matches -- a real, idempotent write

This is not a simulation: it writes `assets` on your file nodes for real, using
`instances.apply` (upsert). Compare the result in Fusion against what you already
hardcoded by hand in Chapter 04 -- they should agree.

In [ ]:
applies = [
    NodeApply(
        space=space,
        external_id=m["source"],
        sources=[NodeOrEdgeData(source=v_file, properties={"assets": [DirectRelationReference(space, m["target"])]})],
    )
    for m in matches
]
if applies:
    client.data_modeling.instances.apply(nodes=applies)
    print(f"applied {len(applies)} file.assets update(s)")
else:
    print("no matches above threshold -- nothing applied")

## Cleanup -- delete the model

`[COMMON MISTAKE]` skipping this because "it's just a training project." The model
lives in a project-wide namespace shared by every participant and every other piece
of work in this project -- always clean up.

In [ ]:
client.entity_matching.delete(id=model.id)

try:
    client.entity_matching.retrieve(external_id=model_xid)
    print("STILL THERE -- delete failed, investigate before moving on")
except Exception:
    print("confirmed deleted")

## Bridge to the Function

You just ran, cell by cell: auth -> list -> fit -> poll -> predict -> apply ->
cleanup. **Now package this into a Cognite Function -- what changes?**

- No interactive login -- the Function runs under its own managed identity
- `space`/file externalIds come from `envVars`, not a notebook variable you set by hand
- The poll deadline becomes a hard budget the Function must respect (no human watching a progress bar)
- `print(...)` becomes a returned, JSON-serializable `dict` -- that's what shows up in the call-result log
- The "drop a leftover model" defensive delete at the top of `fit` matters *more* --
  a Function can be called repeatedly with no human noticing a stale model

Continue to [Chapter 07, section 7.7](../07-entity-matching.md#77-write-the-function-matchdocuments)
to see the actual handler.